# Week 10: K-Means Clustering
## Digital Customer Journey and E-Commerce Retention Analytics

**Student:** Javed Aamer

This analysis looks for natural customer segments based on engagement and transaction behavior.

The clustering model does **not** use `Churn`. Instead, I first group customers using behavioral variables and then compare churn rates across the resulting segments. This keeps the clustering unsupervised while still allowing me to ask whether the segments are useful for retention decisions.

The main questions are how much feature scaling changes the result, what the elbow and Silhouette methods suggest for the number of clusters, and whether the final clusters describe meaningfully different customer profiles.

## Focus for Week 10

The analysis covers:

- k-means clustering;
- within-cluster sum of squares (WCSS/inertia);
- the elbow method;
- Silhouette score;
- feature scaling;
- Euclidean distance and why it matches the k-means centroid update;
- Manhattan and cosine distance as post-hoc diagnostics;
- cluster stability across random initializations;
- customer-segment interpretation and churn profiling.

## 1. Questions guiding the analysis

1. Which customer variables are likely to dominate distance calculations if I do not scale them?
2. How much does scaling change the apparent clustering quality?
3. What number of clusters is suggested by the elbow method?
4. What number of clusters is favored by the Silhouette score?
5. If those two methods disagree, which solution is more useful for the customer-journey problem?
6. What characteristics distinguish the final customer segments?
7. Do the segments differ in churn rate even though churn is not used to build them?
8. Are the clusters stable across different k-means initializations?

In [1]:
# Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

RANDOM_STATE = 42

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
# Load the same cleaned customer-level dataset used in the recent capstone notebooks
candidate_paths = [
    Path('data/churn_clean.csv'),
    Path('churn_clean.csv'),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find churn_clean.csv. Put it in a data/ folder next to this notebook.'
    )

churn = pd.read_csv(DATA_PATH)

print(f'Loaded: {DATA_PATH}')
print(f'Shape: {churn.shape[0]:,} rows x {churn.shape[1]} columns')
print(f'Unique customers: {churn["CustomerID"].nunique():,}')
print(f'Overall churn rate: {churn["Churn"].mean():.2%}')

display(churn.head())

Loaded: data/churn_clean.csv
Shape: 5,630 rows x 20 columns
Unique customers: 5,630
Overall churn rate: 16.84%


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0000,Mobile Phone,3,6.0000,Debit Card,Female,3.0000,3,Laptop & Accessory,2,Single,9,1,11.0000,1.0000,1.0000,5.0000,159.9300
1,50002,1,9.0000,Phone,1,8.0000,UPI,Male,3.0000,4,Mobile,3,Single,7,1,15.0000,0.0000,1.0000,0.0000,120.9000
2,50003,1,9.0000,Phone,1,30.0000,Debit Card,Male,2.0000,4,Mobile,3,Single,6,1,14.0000,0.0000,1.0000,3.0000,120.2800
3,50004,1,0.0000,Phone,3,15.0000,Debit Card,Male,2.0000,4,Laptop & Accessory,5,Single,8,0,23.0000,0.0000,1.0000,3.0000,134.0700
4,50005,1,0.0000,Phone,1,12.0000,CC,Male,3.0000,3,Mobile,5,Single,3,0,11.0000,1.0000,1.0000,3.0000,129.6000


## 2. Variables used for segmentation

I focus the clustering on seven variables that describe customer history, recent activity, incentives, and experience:

- `Tenure`
- `SatisfactionScore`
- `Complain`
- `CouponUsed`
- `OrderCount`
- `DaySinceLastOrder`
- `CashbackAmount`

I intentionally leave `Churn` out of the clustering features. If churn were included, a later comparison of churn rates across clusters would partly reflect information that was already used to create the groups.

I also leave identifiers and categorical fields out of this first k-means analysis. Standard k-means is based on Euclidean geometry, so a focused numeric feature set makes the distance calculation and centroid interpretation easier to understand.

In [3]:
cluster_features = [
    'Tenure',
    'SatisfactionScore',
    'Complain',
    'CouponUsed',
    'OrderCount',
    'DaySinceLastOrder',
    'CashbackAmount',
]

X_raw = churn[cluster_features].copy()

summary = X_raw.describe().T[
    ['mean', 'std', 'min', '25%', '50%', '75%', 'max']
]
display(summary)

,mean,std,min,25%,50%,75%,max
Tenure,10.1341,8.3580,0.0000,3.0000,9.0000,15.0000,61.0000
SatisfactionScore,3.0668,1.3802,1.0000,2.0000,3.0000,4.0000,5.0000
Complain,0.2849,0.4514,0.0000,0.0000,0.0000,1.0000,1.0000
CouponUsed,1.7169,1.8576,0.0000,1.0000,1.0000,2.0000,16.0000
OrderCount,2.9618,2.8792,1.0000,1.0000,2.0000,3.0000,16.0000
DaySinceLastOrder,4.4593,3.5706,0.0000,2.0000,3.0000,7.0000,46.0000
CashbackAmount,177.2230,49.2070,0.0000,145.7700,163.2800,196.3925,324.9900


## 3. Why scaling matters here

Before fitting k-means, I check how much each feature contributes to the raw numerical spread.

This matters because k-means assigns points using distance. A feature measured on a much larger numeric scale can dominate the distance calculation even if it is not more important to the business problem.

My expectation is that `CashbackAmount` will dominate an unscaled Euclidean distance because its numerical variance is much larger than the other variables.

In [4]:
# Share of total raw variance by feature
raw_variance = X_raw.var(ddof=0)
variance_share = (
    raw_variance / raw_variance.sum()
).sort_values(ascending=False)

variance_table = pd.DataFrame({
    'Raw variance': raw_variance,
    'Share of total raw variance': raw_variance / raw_variance.sum(),
}).sort_values('Share of total raw variance', ascending=False)

display(variance_table)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    variance_share.index[::-1],
    (variance_share.values[::-1] * 100),
)
ax.set_xlabel('Share of total raw variance (%)')
ax.set_ylabel('Feature')
ax.set_title('Without scaling, one feature dominates the distance geometry')
ax.grid(axis='x')
plt.tight_layout()
plt.show()

,Raw variance,Share of total raw variance
CashbackAmount,"2,420.9023",0.9617
Tenure,69.8429,0.0277
DaySinceLastOrder,12.7471,0.0051
OrderCount,8.2886,0.0033
CouponUsed,3.4502,0.0014
SatisfactionScore,1.9046,0.0008
Complain,0.2037,0.0001


<Figure size 900x500 with 1 Axes>

`CashbackAmount` contributes about **96% of the total raw variance** across these seven variables. That means an unscaled k-means solution would be driven heavily by cashback differences.

This is exactly the situation where standardization is needed. I therefore use `StandardScaler`, giving each feature mean 0 and standard deviation 1 before fitting the main k-means models.

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

scaled_check = pd.DataFrame(
    X_scaled,
    columns=cluster_features,
).agg(['mean', 'std']).T

display(scaled_check)

,mean,std
Tenure,0.0000,1.0001
SatisfactionScore,-0.0000,1.0001
Complain,-0.0000,1.0001
CouponUsed,-0.0000,1.0001
OrderCount,-0.0000,1.0001
DaySinceLastOrder,-0.0000,1.0001
CashbackAmount,0.0000,1.0001


## 4. Elbow method

K-means minimizes the within-cluster sum of squared Euclidean distances, also called **inertia** or **WCSS**.

As the number of clusters increases, WCSS must decrease because the model has more centroids available. The useful question is where the improvement begins to flatten.

I fit k-means for values of k from 1 through 10 using the standardized features.

In [6]:
ks = list(range(1, 11))
inertias = []

for k in ks:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=30,
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

elbow_results = pd.DataFrame({
    'k': ks,
    'Inertia': inertias,
})
elbow_results['Percent drop from prior k'] = (
    -elbow_results['Inertia'].pct_change() * 100
)

display(elbow_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    elbow_results['k'],
    elbow_results['Inertia'],
    marker='o',
)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Within-cluster sum of squares (inertia)')
ax.set_title('Elbow method for standardized customer features')
ax.set_xticks(ks)
ax.grid()
plt.tight_layout()
plt.show()

,k,Inertia,Percent drop from prior k
0,1,"39,410.0000",NaN
1,2,"30,871.9565",21.6647
2,3,"26,362.9393",14.6055
3,4,"22,777.8944",13.5988
4,5,"20,743.5530",8.9312
5,6,"19,432.0222",6.3226
6,7,"18,273.8911",5.9599
7,8,"17,297.3385",5.3440
8,9,"16,425.5090",5.0403
9,10,"15,583.9084",5.1237


<Figure size 800x500 with 1 Axes>

The largest improvement occurs when moving from one cluster to two. The reductions continue through k = 4 and then become noticeably smaller.

To make the bend less subjective, I also use a simple geometric check: after normalizing the axes, I measure how far each point lies from the straight line joining k = 1 and k = 10. The point farthest from that line is the strongest bend in this range.

In [7]:
# Simple distance-to-line elbow diagnostic
x = elbow_results['k'].to_numpy(dtype=float)
y = elbow_results['Inertia'].to_numpy(dtype=float)

x_norm = (x - x.min()) / (x.max() - x.min())
y_norm = (y - y.min()) / (y.max() - y.min())

# Endpoints are approximately (0, 1) and (1, 0), so the line is x + y - 1 = 0.
elbow_distance = np.abs(x_norm + y_norm - 1) / np.sqrt(2)
elbow_results['Distance from endpoint line'] = elbow_distance

elbow_k = int(elbow_results.loc[
    elbow_results['Distance from endpoint line'].idxmax(),
    'k'
])

print(f'Geometric elbow diagnostic: k = {elbow_k}')
display(elbow_results[['k', 'Inertia', 'Percent drop from prior k', 'Distance from endpoint line']])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    elbow_results['k'],
    elbow_results['Distance from endpoint line'],
    marker='o',
)
ax.axvline(elbow_k, linestyle='--', label=f'Largest bend: k={elbow_k}')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Distance from endpoint line')
ax.set_title('A geometric view of the elbow')
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

Geometric elbow diagnostic: k = 4


,k,Inertia,Percent drop from prior k,Distance from endpoint line
0,1,"39,410.0000",NaN,0.0000
1,2,"30,871.9565",21.6647,0.1748
2,3,"26,362.9393",14.6055,0.2301
3,4,"22,777.8944",13.5988,0.2579
4,5,"20,743.5530",8.9312,0.2397
5,6,"19,432.0222",6.3226,0.2001
6,7,"18,273.8911",5.9599,0.1559
7,8,"17,297.3385",5.3440,0.1063
8,9,"16,425.5090",5.0403,0.0536
9,10,"15,583.9084",5.1237,0.0000


<Figure size 800x500 with 1 Axes>

The geometric check identifies **k = 4** as the clearest bend. That does not automatically make four clusters the best choice, so I compare it with the Silhouette score next.

## 5. Silhouette score

The Silhouette score compares how close each customer is to customers in the same cluster with how close that customer is to the nearest alternative cluster.

Values closer to 1 indicate stronger separation. Values near 0 indicate overlapping clusters.

Unlike the elbow method, the Silhouette score can directly compare candidate values of k, but it may favor a simpler solution if additional business-relevant subgroups overlap in feature space.

In [8]:
silhouette_rows = []
labels_by_k = {}

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=30,
    )
    labels = model.fit_predict(X_scaled)
    labels_by_k[k] = labels
    silhouette_rows.append({
        'k': k,
        'Silhouette score': silhouette_score(
            X_scaled,
            labels,
            metric='euclidean',
        ),
    })

silhouette_results = pd.DataFrame(silhouette_rows)
best_silhouette_k = int(
    silhouette_results.loc[
        silhouette_results['Silhouette score'].idxmax(),
        'k'
    ]
)

print(f'Highest Silhouette score: k = {best_silhouette_k}')
display(silhouette_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    silhouette_results['k'],
    silhouette_results['Silhouette score'],
    marker='o',
)
ax.axvline(
    best_silhouette_k,
    linestyle='--',
    label=f'Highest Silhouette: k={best_silhouette_k}',
)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Mean Silhouette score')
ax.set_title('Silhouette score across candidate cluster counts')
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

Highest Silhouette score: k = 2


,k,Silhouette score
0,2,0.2945
1,3,0.2326
2,4,0.2427
3,5,0.2077
4,6,0.2077
5,7,0.2092
6,8,0.2053
7,9,0.2030
8,10,0.2097


<Figure size 800x500 with 1 Axes>

### Elbow and Silhouette do not select the same k

The Silhouette score is highest at **k = 2**, while the elbow analysis points to **k = 4**.

That disagreement is useful rather than problematic. A two-cluster solution gives the cleanest separation, but it compresses the customer population into broad groups. Four clusters provide more detail for retention and engagement decisions while still retaining a positive Silhouette score.

For the exploratory customer-segmentation analysis, I carry **k = 4** forward. I treat that as a business-granularity choice supported by the elbow, not as a claim that k = 4 is mathematically superior on every criterion.

## 6. Scaling changes the apparent clustering quality

To see why the scaling decision matters, I repeat the Silhouette comparison on the raw features.

A higher raw-data Silhouette score would not automatically mean the unscaled clusters are better. If one feature dominates the distance calculation, k-means can create apparently well-separated groups that mostly reflect that one measurement scale.

In [9]:
raw_silhouette_rows = []
scaled_silhouette_rows = []

for k in range(2, 7):
    raw_model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=30,
    )
    raw_labels = raw_model.fit_predict(X_raw)
    raw_silhouette_rows.append({
        'k': k,
        'Silhouette': silhouette_score(X_raw, raw_labels),
        'Scale': 'Raw features',
    })

    scaled_model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=30,
    )
    scaled_labels = scaled_model.fit_predict(X_scaled)
    scaled_silhouette_rows.append({
        'k': k,
        'Silhouette': silhouette_score(X_scaled, scaled_labels),
        'Scale': 'Standardized features',
    })

scale_silhouette = pd.DataFrame(
    raw_silhouette_rows + scaled_silhouette_rows
)

display(scale_silhouette)

fig, ax = plt.subplots(figsize=(8, 5))
for label, group in scale_silhouette.groupby('Scale'):
    group = group.sort_values('k')
    ax.plot(
        group['k'],
        group['Silhouette'],
        marker='o',
        label=label,
    )

ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.set_title('Raw features can look more separated because scale dominates distance')
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

,k,Silhouette,Scale
0,2,0.6517,Raw features
1,3,0.5253,Raw features
2,4,0.4496,Raw features
3,5,0.4325,Raw features
4,6,0.4069,Raw features
5,2,0.2945,Standardized features
6,3,0.2326,Standardized features
7,4,0.2427,Standardized features
8,5,0.2077,Standardized features
9,6,0.2077,Standardized features


<Figure size 800x500 with 1 Axes>

The raw-data Silhouette scores are much higher, but the earlier variance check shows why that result is misleading: `CashbackAmount` contributes about 96% of the raw variance.

The standardized solution is harder to separate because all seven customer behaviors are allowed to contribute on comparable scales. For this project, that is the more meaningful clustering problem.

## 7. Final four-cluster solution

I now fit k-means with k = 4 using the standardized features. The cluster IDs themselves have no business meaning, so I profile each group before assigning descriptive names.

In [10]:
final_k = 4
final_kmeans = KMeans(
    n_clusters=final_k,
    random_state=RANDOM_STATE,
    n_init=50,
)
cluster_id = final_kmeans.fit_predict(X_scaled)

clustered = churn.copy()
clustered['ClusterID'] = cluster_id

profile = clustered.groupby('ClusterID').agg(
    Size=('CustomerID', 'size'),
    ChurnRate=('Churn', 'mean'),
    Tenure=('Tenure', 'mean'),
    SatisfactionScore=('SatisfactionScore', 'mean'),
    Complain=('Complain', 'mean'),
    CouponUsed=('CouponUsed', 'mean'),
    OrderCount=('OrderCount', 'mean'),
    DaySinceLastOrder=('DaySinceLastOrder', 'mean'),
    CashbackAmount=('CashbackAmount', 'mean'),
)

# Assign readable names from the dominant profile characteristics.
complaint_cluster = int(profile['Complain'].idxmax())
loyal_cluster = int(profile['Tenure'].idxmax())
high_order_cluster = int(
    profile.drop(index=[complaint_cluster, loyal_cluster])['OrderCount'].idxmax()
)
remaining_cluster = int(
    list(set(profile.index) - {complaint_cluster, loyal_cluster, high_order_cluster})[0]
)

cluster_names = {
    complaint_cluster: 'Complaint-heavy / high-risk',
    loyal_cluster: 'Long-tenure / high-cashback',
    high_order_cluster: 'High-order / incentive-active',
    remaining_cluster: 'Low-activity / non-complaining',
}

profile['Segment'] = [cluster_names[i] for i in profile.index]
clustered['Segment'] = clustered['ClusterID'].map(cluster_names)

display(profile[['Segment', 'Size', 'ChurnRate', 'Tenure', 'SatisfactionScore', 'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']])

,Segment,Size,ChurnRate,Tenure,SatisfactionScore,Complain,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
ClusterID,,,,,,,,,,
0,High-order / incentive-active,753,0.1501,12.0040,3.1620,0.2683,5.0345,8.7689,8.0558,200.5018
1,Low-activity / non-complaining,2783,0.1351,7.1125,3.1013,0.0000,1.1276,1.9109,3.5282,156.1146
2,Long-tenure / high-cashback,897,0.0256,21.6890,2.9799,0.2285,1.4838,2.6321,5.9331,246.6456
3,Complaint-heavy / high-risk,1197,0.3642,7.3241,2.9916,1.0000,1.1746,1.9992,3.2573,159.6322


The four groups are meaningfully different:

- **Complaint-heavy / high-risk:** every customer in this segment has a complaint indicator of 1, and the post-hoc churn rate is the highest.
- **Long-tenure / high-cashback:** the longest-tenured group, with high cashback and the lowest observed churn rate.
- **High-order / incentive-active:** substantially more orders and coupon use than the other groups.
- **Low-activity / non-complaining:** the largest segment, with relatively low order and coupon activity and no complaint indicator.

These labels are descriptive summaries of the observed cluster profiles, not predefined classes.

## 8. Cluster size

Before interpreting a segment operationally, I want to know whether it represents a meaningful part of the customer base or only a very small group.

In [11]:
segment_sizes = (
    clustered['Segment']
    .value_counts()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(segment_sizes.index, segment_sizes.values)
ax.set_xlabel('Number of customers')
ax.set_ylabel('Segment')
ax.set_title('Customer count by k-means segment')
ax.grid(axis='x')
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

The low-activity group is the largest segment, while the high-order group is the smallest. None of the four segments is so small that its profile is driven by only a handful of customers.

## 9. Cluster profiles on a common scale

Raw means are difficult to compare because the variables have different units. I therefore plot each cluster's mean standardized feature value.

Values above zero are above the full-sample mean; values below zero are below it.

In [12]:
scaled_frame = pd.DataFrame(
    X_scaled,
    columns=cluster_features,
)
scaled_frame['ClusterID'] = cluster_id
scaled_profile = scaled_frame.groupby('ClusterID')[cluster_features].mean()
scaled_profile.index = [cluster_names[i] for i in scaled_profile.index]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(scaled_profile.values, aspect='auto')

ax.set_xticks(np.arange(len(cluster_features)))
ax.set_xticklabels(cluster_features, rotation=45, ha='right')
ax.set_yticks(np.arange(len(scaled_profile.index)))
ax.set_yticklabels(scaled_profile.index)
ax.set_title('Standardized customer profile by segment')

for i in range(scaled_profile.shape[0]):
    for j in range(scaled_profile.shape[1]):
        ax.text(
            j,
            i,
            f'{scaled_profile.iloc[i, j]:.2f}',
            ha='center',
            va='center',
            fontsize=8,
        )

fig.colorbar(im, ax=ax, label='Mean standardized value')
plt.tight_layout()
plt.show()

<Figure size 1000x600 with 2 Axes>

The heatmap makes the segmentation logic visible. The complaint-heavy group is defined mainly by complaints, the long-tenure group by tenure and cashback, and the high-order group by order count and coupon use.

## 10. Churn was not used to cluster customers

Now that the clusters are fixed, I use `Churn` only as a post-hoc outcome to see whether the behavioral segments differ in retention risk.

This is important because any difference in churn rate was not directly forced into the clustering algorithm.

In [13]:
churn_by_segment = (
    clustered.groupby('Segment')['Churn']
    .agg(['mean', 'size'])
    .rename(columns={'mean': 'Churn rate', 'size': 'Customers'})
    .sort_values('Churn rate')
)

display(churn_by_segment)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    churn_by_segment.index,
    churn_by_segment['Churn rate'] * 100,
)
ax.set_xlabel('Observed churn rate (%)')
ax.set_ylabel('Segment')
ax.set_title('Churn differs sharply across clusters even though churn was not an input')
ax.grid(axis='x')
plt.tight_layout()
plt.show()

,Churn rate,Customers
Segment,,
Long-tenure / high-cashback,0.0256,897
Low-activity / non-complaining,0.1351,2783
High-order / incentive-active,0.1501,753
Complaint-heavy / high-risk,0.3642,1197


<Figure size 900x500 with 1 Axes>

The post-hoc result is operationally important. The complaint-heavy segment has a churn rate of roughly **36%**, compared with about **3%** in the long-tenure/high-cashback segment.

For the broader customer-journey problem, this suggests that the unsupervised segmentation is capturing behavior that is strongly associated with retention risk. I would prioritize the complaint-heavy segment for service recovery or retention analysis, while treating the long-tenure group as a useful reference for loyal-customer behavior.

This is still an association, not evidence that the cluster characteristics cause churn.

## 11. Visualizing the clusters in two dimensions

The model uses seven dimensions, which cannot be plotted directly. I use PCA only for visualization, reducing the standardized data to two components after the clustering is complete.

The PCA coordinates do not determine the clusters; they provide a two-dimensional view of the existing k-means assignments.

In [14]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_frame = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Segment': clustered['Segment'].to_numpy(),
})

fig, ax = plt.subplots(figsize=(9, 6))
for segment, group in pca_frame.groupby('Segment'):
    ax.scatter(
        group['PC1'],
        group['PC2'],
        s=14,
        alpha=0.45,
        label=segment,
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Two-dimensional view of the four customer segments')
ax.legend(fontsize=8)
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x600 with 1 Axes>

The PCA view shows overlap between groups, which is consistent with the moderate Silhouette score. These are not four perfectly separated populations. The value of the solution is that it identifies interpretable behavioral tendencies within a continuous customer population.

## 12. Selected feature distributions by cluster

The profile means summarize the groups, but distributions show whether those differences are broad or driven by a small number of extreme observations.

In [15]:
# Tenure distribution by segment
segment_order = profile.sort_values('Tenure')['Segment'].tolist()
values = [
    clustered.loc[clustered['Segment'] == s, 'Tenure'].to_numpy()
    for s in segment_order
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(values, tick_labels=segment_order, showfliers=False)
ax.set_ylabel('Tenure')
ax.set_xlabel('Segment')
ax.set_title('Tenure distribution by customer segment')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

In [16]:
# OrderCount distribution by segment
segment_order = profile.sort_values('OrderCount')['Segment'].tolist()
values = [
    clustered.loc[clustered['Segment'] == s, 'OrderCount'].to_numpy()
    for s in segment_order
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(values, tick_labels=segment_order, showfliers=False)
ax.set_ylabel('OrderCount')
ax.set_xlabel('Segment')
ax.set_title('Order-count distribution by customer segment')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

In [17]:
# CashbackAmount distribution by segment
segment_order = profile.sort_values('CashbackAmount')['Segment'].tolist()
values = [
    clustered.loc[clustered['Segment'] == s, 'CashbackAmount'].to_numpy()
    for s in segment_order
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(values, tick_labels=segment_order, showfliers=False)
ax.set_ylabel('CashbackAmount')
ax.set_xlabel('Segment')
ax.set_title('Cashback distribution by customer segment')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

The distribution plots reinforce the profile interpretation. The long-tenure group is shifted toward longer relationships and higher cashback, while the incentive-active group is clearly shifted toward larger order counts.

## 13. Distance metrics

Standard k-means is built around **squared Euclidean distance**. The centroid is the arithmetic mean because the mean minimizes the sum of squared Euclidean distances within a cluster.

Manhattan distance follows different geometry: the coordinate-wise median, rather than the mean, minimizes the sum of absolute distances. For that reason, simply replacing Euclidean distance with Manhattan distance would no longer be standard k-means.

I still calculate Silhouette scores for the fixed four-cluster labels under Euclidean, Manhattan, and cosine distance. These are **diagnostics of the same cluster assignments**, not three different k-means models.

In [18]:
distance_scores = pd.DataFrame({
    'Distance metric': ['Euclidean', 'Manhattan', 'Cosine'],
    'Silhouette score': [
        silhouette_score(X_scaled, cluster_id, metric='euclidean'),
        silhouette_score(X_scaled, cluster_id, metric='manhattan'),
        silhouette_score(X_scaled, cluster_id, metric='cosine'),
    ],
})

display(distance_scores)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    distance_scores['Distance metric'],
    distance_scores['Silhouette score'],
)
ax.set_ylabel('Silhouette score for fixed k=4 labels')
ax.set_xlabel('Distance metric used for evaluation')
ax.set_title('The same clusters look different under different distance geometries')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

,Distance metric,Silhouette score
0,Euclidean,0.2427
1,Manhattan,0.2630
2,Cosine,0.3411


<Figure size 700x500 with 1 Axes>

Cosine distance gives the highest post-hoc Silhouette score for these fixed labels, followed by Manhattan and Euclidean distance. That does **not** mean cosine should replace Euclidean distance inside standard k-means. The model was fitted by minimizing Euclidean WCSS, so the alternative metrics only show that the apparent separation depends on how similarity is defined.

## 14. Mean versus median under Euclidean and Manhattan distance

To make the distance-metric issue concrete, I use the complaint-heavy cluster and compare its arithmetic mean with its coordinate-wise median.

If the course explanation is correct, the mean should have the lower squared-Euclidean objective, while the median should have the lower Manhattan objective.

In [19]:
example_cluster = complaint_cluster
X_example = X_raw.loc[cluster_id == example_cluster].to_numpy(dtype=float)

mean_center = X_example.mean(axis=0)
median_center = np.median(X_example, axis=0)

sse_mean = ((X_example - mean_center) ** 2).sum()
sse_median = ((X_example - median_center) ** 2).sum()
l1_mean = np.abs(X_example - mean_center).sum()
l1_median = np.abs(X_example - median_center).sum()

center_comparison = pd.DataFrame({
    'Objective': ['Squared Euclidean', 'Squared Euclidean', 'Manhattan', 'Manhattan'],
    'Center': ['Mean', 'Median', 'Mean', 'Median'],
    'Objective value': [sse_mean, sse_median, l1_mean, l1_median],
})

display(center_comparison)

relative = pd.DataFrame({
    'Center / objective': [
        'Mean - squared Euclidean',
        'Median - squared Euclidean',
        'Mean - Manhattan',
        'Median - Manhattan',
    ],
    'Relative objective': [
        sse_mean / min(sse_mean, sse_median),
        sse_median / min(sse_mean, sse_median),
        l1_mean / min(l1_mean, l1_median),
        l1_median / min(l1_mean, l1_median),
    ],
})

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    relative['Center / objective'][::-1],
    relative['Relative objective'][::-1],
)
ax.axvline(1.0, linestyle='--')
ax.set_xlabel('Objective relative to the best center for that distance')
ax.set_title('Mean minimizes squared Euclidean distance; median minimizes Manhattan distance')
ax.grid(axis='x')
plt.tight_layout()
plt.show()

,Objective,Center,Objective value
0,Squared Euclidean,Mean,"1,122,010.8963"
1,Squared Euclidean,Median,"1,162,828.0244"
2,Manhattan,Mean,"39,845.4937"
3,Manhattan,Median,"39,012.2200"


<Figure size 900x500 with 1 Axes>

The result matches the geometry behind k-means: the arithmetic mean gives the smaller squared-Euclidean objective, while the coordinate-wise median gives the smaller Manhattan objective.

That is why Euclidean distance is not an arbitrary default in k-means; it is tied directly to the centroid update that defines the algorithm.

## 15. Cluster stability across random initializations

K-means can converge to different local solutions depending on its starting centroids. I use `n_init` to reduce that risk, but I also test whether the four-cluster assignments remain similar across several random seeds.

Adjusted Rand Index (ARI) compares two clusterings while ignoring the arbitrary numeric cluster labels. A value close to 1 means the assignments are almost identical.

In [20]:
reference_labels = final_kmeans.labels_
stability_rows = []

for seed in range(10):
    model = KMeans(
        n_clusters=4,
        random_state=seed,
        n_init=20,
    )
    candidate_labels = model.fit_predict(X_scaled)
    stability_rows.append({
        'Random seed': seed,
        'Adjusted Rand Index vs. reference': adjusted_rand_score(
            reference_labels,
            candidate_labels,
        ),
    })

stability = pd.DataFrame(stability_rows)
display(stability)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    stability['Random seed'],
    stability['Adjusted Rand Index vs. reference'],
    marker='o',
)
ax.set_ylim(0, 1.02)
ax.set_xlabel('Random seed')
ax.set_ylabel('Adjusted Rand Index')
ax.set_title('The four-cluster solution is stable across random initializations')
ax.grid()
plt.tight_layout()
plt.show()

,Random seed,Adjusted Rand Index vs. reference
0,0,0.9956
1,1,0.9934
2,2,0.9919
3,3,0.9951
4,4,0.9959
5,5,0.9992
6,6,0.9988
7,7,0.9959
8,8,0.9959
9,9,0.9988


<Figure size 800x500 with 1 Axes>

The ARI values are all approximately **0.99 or higher**, so the four-cluster solution is very stable across these random initializations. That makes the cluster profiles less likely to be an artifact of one unlucky centroid initialization.

## 16. Expected and unexpected results

### What I expected

I expected scaling to matter because `CashbackAmount` is measured on a much larger numeric scale than variables such as `Complain` or `SatisfactionScore`. I also expected a complaint-heavy customer group to show elevated churn when I profiled the clusters afterward, because complaints had repeatedly appeared as a churn signal in earlier supervised models.

### What surprised me

The unscaled data produced much higher Silhouette scores than the standardized data. At first glance that could look like better clustering, but the variance analysis showed that the raw geometry was dominated by `CashbackAmount`. The apparently cleaner separation was therefore not the multi-feature customer segmentation I wanted.

I also did not expect the elbow and Silhouette methods to disagree as clearly as they did. Silhouette favored two broad clusters, while the elbow supported four. That forced a useful modeling decision: prioritize maximum geometric separation with k = 2, or accept some overlap in exchange for more actionable customer segments with k = 4.

## 17. What this means for the customer-journey problem

The four-cluster model does not replace the supervised churn models from earlier weeks. It answers a different question: **what kinds of customers naturally appear in the behavioral data before I ask whether they churn?**

The strongest business finding is the contrast between the complaint-heavy segment and the long-tenure/high-cashback segment. Their observed churn rates differ sharply even though churn was not used to create the clusters. This gives the retention problem a segmentation layer:

- complaint-heavy customers are a natural group for service recovery and retention outreach;
- long-tenure/high-cashback customers provide a reference profile for loyalty;
- high-order/incentive-active customers may deserve separate promotion and value analysis;
- the large low-activity segment may require additional behavioral data before it can be divided meaningfully.

The moderate Silhouette score also matters. I would not describe these as four perfectly separated customer types. They are overlapping behavioral segments that can support prioritization and further analysis.

## 18. Limitations

- K-means assumes Euclidean, roughly spherical cluster geometry.
- The final k = 4 choice balances elbow evidence and business interpretability; the highest Silhouette score occurs at k = 2.
- The seven input variables include a binary complaint indicator and several discrete counts, so the Euclidean representation is an approximation.
- Categorical customer attributes are not included in this first k-means model.
- Standardization gives each feature equal variance, which is a modeling choice rather than proof that every variable deserves equal business weight.
- Churn differences across clusters are post-hoc associations, not causal effects.
- PCA is used only for visualization and does not preserve all seven dimensions.

## 19. Reproducibility notes

- Random seed: 42
- Data file: `data/churn_clean.csv`
- Customers: 5,630
- Clustering inputs: 7 numeric customer behavior / experience variables
- `Churn`: excluded from clustering and used only for post-hoc profiling
- Scaling: `StandardScaler`
- Candidate k values for elbow: 1 through 10
- Candidate k values for Silhouette: 2 through 10
- Main k-means distance: squared Euclidean
- Final exploratory solution: k = 4
- `n_init`: 50 for the final fit
- Alternative distance metrics: used only to evaluate the fixed labels, not to refit standard k-means

## Reference for further reading

A useful external reference for this analysis is the scikit-learn documentation for `KMeans` and clustering evaluation. It clearly documents the inertia objective, initialization behavior, and Silhouette analysis used here:

**scikit-learn.** *Clustering: K-means and Silhouette analysis.* https://scikit-learn.org/stable/modules/clustering.html

I would use this as the Week 10 Yellowdig resource because it is the primary documentation for the implementation used in the notebook and connects the algorithm directly to the evaluation measures.